In [ ]:
import pandas as pd
import kagglehub as kh
import os
import dotenv

In [ ]:
#load kaggle key from env file
dotenv.load_dotenv()

print(os.getenv("KAGGLE_USERNAME"))
print(os.getenv("KAGGLE_KEY"))

In [ ]:
#Download public synthetic upcoding health insurance claims dataset
handle = "apexsyntheticdata/synthetic-healthcare-fraud-dataset-10k"
path =  kh.dataset_download(handle)

print(path)

In [ ]:
#Access downloaded dataset
for file in os.listdir(path):
    if(file.endswith(".csv")):
        csv_read = os.path.join(path, file)
        df = pd.read_csv(csv_read)

print(df.shape)
print(df.head())

Explore dataset

In [ ]:
print("-" * 60)
print("\t\tEXPLORATORY DATA ANALYSIS")
print("-" * 60)

print(df.shape)
print(df.columns.tolist())
print("-" * 60)

#print(df.head())
print("-" * 60)

print(df["is_anomaly"].value_counts())
print(df["anomaly_type"].value_counts())
print("-" * 60)


Preprocess the Dataset

In [ ]:
#Keep labels for evaluation. It should not be used for training
X = df.drop(["is_anomaly", "anomaly_type"], axis=1) #features only
y = df["is_anomaly"] #Hidden labels for testing

#Encode categorical variables (diagnosis codes, procedures codes, etc)
from sklearn.preprocessing import  LabelEncoder
categorical_cols = X.select_dtypes(include = ["object"]).columns

for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

#scale numerical features
from sklearn.preprocessing import StandardScaler
num_cols = X.select_dtypes(include=["float64","int64"]).columns
X[num_cols] = StandardScaler().fit_transform(X[num_cols])

In [12]:
from sklearn.ensemble import IsolationForest

iso_forest =  IsolationForest(
    n_estimators = 100,
    contamination = 0.05,
    random_state = 42
)

iso_forest.fit(X)
y_pred = iso_forest.predict(X)

scores = iso_forest.score_samples(X)

print(scores)

[-0.4926097  -0.52088954 -0.49957255 ... -0.47071398 -0.56961184
 -0.50743707]
